<a href="https://colab.research.google.com/github/waelantar/ATTS_Complete_Free_Package/blob/main/01_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NanoLM-Derja V4: Notebook 1 - Data Preparation & Cleaning

**Purpose**: Download, clean, and prepare all Tunisian Derja datasets with harsh filtering.

**Run on**: Google Colab (single T4) to preserve Kaggle compute for training.

**Datasets**:
- `hamzabouajila/tunisian-derja-unified-raw-corpus` (primary)
- `linagora/Tunisian_Derja_Dataset` (secondary)
- `fbougares/TEDxTN` (high-quality)
- `chaymafourati/tunizi` (Arabizi)
- `arbml/Tunisian_Dialect_Corpus` (tweets)
- `khaled123/Tunisian_Dialectic_English_Derja` (harsh filtering required)

**Output**: Clean train/val splits with register tags (~400k-600k samples)

## 1. Setup & Installation

In [1]:
# Install required packages
!pip install -q datasets pandas pyarrow datasketch tqdm huggingface_hub langdetect

import os
import re
import json
import random
import hashlib
from typing import List, Dict, Set, Optional, Tuple
from collections import Counter
from dataclasses import dataclass
from tqdm.auto import tqdm

import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict
from datasketch import MinHash, MinHashLSH

# Set seeds for reproducibility
random.seed(42)

print("✅ All packages loaded successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 23.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 9.3 MB/s eta 0:00:00
✅ All packages loaded successfully!


## 2. Configuration

In [2]:
@dataclass
class DataConfig:
    """Data processing configuration."""
    # Output paths
    OUTPUT_DIR: str = "derja_data_v4"
    TRAIN_FILE: str = "train.jsonl"
    VAL_FILE: str = "val.jsonl"

    # Filtering thresholds
    MIN_TEXT_LENGTH: int = 20
    MAX_TEXT_LENGTH: int = 2000
    MIN_ARABIC_RATIO: float = 0.15  # Allow code-switching
    MAX_MSA_PATTERNS: int = 2  # Allow some MSA but not pure MSA

    # Deduplication
    MINHASH_THRESHOLD: float = 0.85
    MINHASH_NUM_PERM: int = 128

    # khaled123 specific limits
    KHALED123_MAX_SAMPLES: int = 80000
    KHALED123_MIN_ARABIC_RATIO: float = 0.40  # Stricter for this dataset

    # Split ratio
    VAL_RATIO: float = 0.05  # 5% validation

    # Target total samples
    TARGET_MIN_SAMPLES: int = 400000
    TARGET_MAX_SAMPLES: int = 600000

config = DataConfig()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
print(f"📁 Output directory: {config.OUTPUT_DIR}")

📁 Output directory: derja_data_v4


## 3. NSFW/Offensive Content Blocklist

**Critical for khaled123 dataset** - Contains unethical content that must be removed.

In [3]:
# Comprehensive blocklist for NSFW/offensive content
# Includes Arabic, Arabizi, French, and English terms

NSFW_BLOCKLIST = {
    # Arabic script offensive terms
    'سكس', 'نيك', 'زب', 'كس', 'شرموط', 'قحب', 'عاهر', 'متناك', 'طيز',
    'منيوك', 'زاني', 'لوطي', 'خول', 'بوزبال', 'كلب', 'حمار', 'خرا',
    'اباحي', 'اباحية', 'بورن', 'اغتصاب', 'تحرش',

    # Arabizi offensive terms
    'nik', 'nayyek', 'nayek', 'zebi', 'zeb', 'za3ma', 'kahba', 'charmouta',
    '9a7ba', 'manyouk', 'kos', 'tiz', '5ara', 'kalb', '7mar', 'bouzbal',

    # French offensive terms
    'putain', 'salope', 'connard', 'enculer', 'merde', 'bordel', 'pute',
    'nique', 'baiser', 'foutre',

    # English offensive/NSFW terms
    'fuck', 'shit', 'bitch', 'ass', 'dick', 'cock', 'pussy', 'whore',
    'porn', 'xxx', 'sex', 'nude', 'naked', 'horny', 'slut', 'cum',
    'nigger', 'nigga', 'faggot', 'retard',

    # Violence/hate terms
    'kill', 'murder', 'terrorist', 'bomb', 'suicide', 'rape',
    'ارهاب', 'قتل', 'انتحار', 'داعش',

    # Spam indicators
    'click here', 'free money', 'winner', 'congratulations',
    '+18', '18+', 'adult only',
}

# Compile regex patterns for efficiency
NSFW_PATTERNS = [re.compile(rf'\b{re.escape(term)}\b', re.IGNORECASE)
                 for term in NSFW_BLOCKLIST]

print(f"🚫 Loaded {len(NSFW_BLOCKLIST)} offensive terms in blocklist")

🚫 Loaded 85 offensive terms in blocklist


In [4]:
# MSA (Modern Standard Arabic) patterns - too formal, not Derja
MSA_PATTERNS = [
    r'\bالذي\b', r'\bالتي\b', r'\bاللذان\b', r'\bاللتان\b',
    r'\bإنّ\b', r'\bأنّ\b', r'\bلكنّ\b',
    r'\bسوف\b', r'\bلماذا\b', r'\bماذا\b',
    r'\bهذان\b', r'\bهاتان\b', r'\bهؤلاء\b',
    r'\bحيث\b', r'\bإذ\b', r'\bبينما\b',
]
MSA_COMPILED = [re.compile(p) for p in MSA_PATTERNS]

# Derja marker words (presence indicates authentic Tunisian)
DERJA_MARKERS = [
    'برشا', 'ياخي', 'باهي', 'شنوة', 'علاش', 'كيفاش', 'شكون',
    'هكا', 'هكاكا', 'توا', 'فيسع', 'مافماش', 'فما', 'ثمة',
    'نحب', 'نمشي', 'نجي', 'بش', 'باش', 'مش', 'موش',
    'عندي', 'عندك', 'كان', 'لوكان', 'خاطر', 'على خاطر',
    'وقتاش', 'وينو', 'وينها', 'شبيك', 'شبيها', 'عيشك',
]

print(f"📝 Loaded {len(MSA_PATTERNS)} MSA patterns and {len(DERJA_MARKERS)} Derja markers")

📝 Loaded 16 MSA patterns and 33 Derja markers


## 4. Text Cleaning Class

In [5]:
class DerjaTextCleaner:
    """Comprehensive text cleaner for Tunisian Derja."""

    def __init__(self, config: DataConfig, harsh_mode: bool = False):
        self.config = config
        self.harsh_mode = harsh_mode  # Extra strict for khaled123
        self.stats = Counter()

        # Compile patterns
        self.url_pattern = re.compile(r'https?://\S+|www\.\S+')
        self.mention_pattern = re.compile(r'@[\w_]+')
        self.hashtag_pattern = re.compile(r'#\w+')
        self.email_pattern = re.compile(r'\S+@\S+\.\S+')
        self.emoji_pattern = re.compile(
            "[\U0001F600-\U0001F9FF"
            "\U0001F300-\U0001F5FF"
            "\U0001F680-\U0001F6FF"
            "\U0001F1E0-\U0001F1FF"
            "\U00002702-\U000027B0]+"
        )
        self.repeated_char_pattern = re.compile(r'(.)\1{3,}')  # 4+ repeated chars
        self.repeated_word_pattern = re.compile(r'\b(\w+)\s+\1\b', re.IGNORECASE)

        # Arabic character range
        self.arabic_pattern = re.compile(r'[\u0600-\u06FF\u0750-\u077F]')
        # Arabizi number-letters
        self.arabizi_nums = set('3579')

    def _count_arabic_chars(self, text: str) -> int:
        """Count Arabic script characters."""
        return len(self.arabic_pattern.findall(text))

    def _count_arabizi_nums(self, text: str) -> int:
        """Count Arabizi number-letters (3, 5, 7, 9)."""
        return sum(1 for c in text if c in self.arabizi_nums)

    def _has_nsfw(self, text: str) -> bool:
        """Check for NSFW/offensive content."""
        text_lower = text.lower()
        for pattern in NSFW_PATTERNS:
            if pattern.search(text_lower):
                return True
        return False

    def _count_msa_patterns(self, text: str) -> int:
        """Count MSA (formal Arabic) patterns."""
        return sum(1 for p in MSA_COMPILED if p.search(text))

    def _has_derja_markers(self, text: str) -> bool:
        """Check for Tunisian Derja marker words."""
        return any(marker in text for marker in DERJA_MARKERS)

    def _is_repetitive(self, text: str) -> bool:
        """Check for repetitive patterns."""
        words = text.split()
        if len(words) < 3:
            return False

        # Check word repetition ratio
        word_counts = Counter(words)
        most_common_count = word_counts.most_common(1)[0][1]
        if most_common_count >= 4 and most_common_count / len(words) > 0.3:
            return True

        # Check for exact phrase repetition
        if self.repeated_word_pattern.search(text):
            # Allow if it's a single occurrence
            matches = self.repeated_word_pattern.findall(text)
            if len(matches) >= 2:
                return True

        return False

    def clean(self, text: str) -> Optional[str]:
        """Clean text and return None if should be filtered."""
        if not text or not isinstance(text, str):
            self.stats['empty'] += 1
            return None

        original_text = text

        # === STEP 1: Basic cleaning ===
        text = self.url_pattern.sub(' ', text)
        text = self.email_pattern.sub(' ', text)
        text = self.mention_pattern.sub(' ', text)

        # Remove hashtags but keep the word
        text = self.hashtag_pattern.sub(lambda m: m.group()[1:], text)

        # Remove excessive emojis
        emoji_matches = self.emoji_pattern.findall(text)
        if len(emoji_matches) > 5:
            self.stats['emoji_spam'] += 1
            return None
        text = self.emoji_pattern.sub(' ', text)

        # Normalize repeated characters (keeeep -> keep)
        text = self.repeated_char_pattern.sub(r'\1\1', text)

        # Normalize whitespace
        text = ' '.join(text.split())

        # === STEP 2: Length filter ===
        if len(text) < self.config.MIN_TEXT_LENGTH:
            self.stats['too_short'] += 1
            return None

        if len(text) > self.config.MAX_TEXT_LENGTH:
            text = text[:self.config.MAX_TEXT_LENGTH]

        # === STEP 3: NSFW filter (CRITICAL) ===
        if self._has_nsfw(text):
            self.stats['nsfw'] += 1
            return None

        # === STEP 4: Arabic/Derja ratio check ===
        arabic_chars = self._count_arabic_chars(text)
        arabizi_nums = self._count_arabizi_nums(text)
        total_chars = len(text.replace(' ', ''))

        if total_chars == 0:
            self.stats['empty_after_clean'] += 1
            return None

        # Calculate "Derja score": Arabic chars + Arabizi numbers
        derja_ratio = (arabic_chars + arabizi_nums * 3) / total_chars  # Weight Arabizi nums

        min_ratio = (self.config.KHALED123_MIN_ARABIC_RATIO if self.harsh_mode
                     else self.config.MIN_ARABIC_RATIO)

        if derja_ratio < min_ratio:
            # Exception: If it has clear Arabizi patterns, keep it
            if arabizi_nums >= 2 and len(text) > 30:
                pass  # Keep Arabizi text
            else:
                self.stats['low_arabic'] += 1
                return None

        # === STEP 5: MSA contamination check ===
        msa_count = self._count_msa_patterns(text)
        if msa_count > self.config.MAX_MSA_PATTERNS:
            self.stats['msa'] += 1
            return None

        # === STEP 6: Repetition check ===
        if self._is_repetitive(text):
            self.stats['repetitive'] += 1
            return None

        # === STEP 7: Harsh mode extra checks ===
        if self.harsh_mode:
            # Require at least some Derja markers OR strong Arabic presence
            has_markers = self._has_derja_markers(text)
            if not has_markers and arabic_chars < len(text) * 0.3:
                self.stats['no_derja_markers'] += 1
                return None

            # Filter pure English sentences
            words = text.split()
            english_words = sum(1 for w in words if w.isascii() and w.isalpha() and len(w) > 2)
            if len(words) > 5 and english_words / len(words) > 0.7:
                self.stats['too_english'] += 1
                return None

        self.stats['kept'] += 1
        return text

    def report(self) -> None:
        """Print cleaning statistics."""
        total = sum(self.stats.values())
        kept = self.stats.get('kept', 0)

        print("\n" + "=" * 50)
        print("CLEANING STATISTICS")
        print("=" * 50)

        for key, count in self.stats.most_common():
            pct = count / total * 100 if total > 0 else 0
            emoji = "✅" if key == 'kept' else "❌"
            print(f"  {emoji} {key}: {count:,} ({pct:.1f}%)")

        print("-" * 50)
        print(f"  Total processed: {total:,}")
        print(f"  Keep rate: {kept/total*100:.1f}%" if total > 0 else "  Keep rate: N/A")

print("✅ DerjaTextCleaner class defined")

✅ DerjaTextCleaner class defined


## 5. Register Tagging

In [6]:
# Register definitions
REGISTERS = {
    0: "GENZ_ARABIZI",   # Latin script with number-letters (3, 7, 9)
    1: "MIXED",          # Code-switched Arabic + Latin
    2: "ARABIC_DOM",     # Arabic script dominant
    3: "UNKNOWN",        # Cannot determine
}

def detect_register(text: str) -> int:
    """
    Detect the linguistic register of Tunisian Derja text.

    Returns:
        0: GENZ_ARABIZI - Contains Arabizi number-letters
        1: MIXED - Code-switched Arabic + Latin
        2: ARABIC_DOM - Predominantly Arabic script
        3: UNKNOWN - Cannot determine
    """
    if not text:
        return 3

    # Count character types
    arabic_chars = sum(1 for c in text if '\u0600' <= c <= '\u06FF')
    latin_chars = sum(1 for c in text if c.isalpha() and ord(c) < 128)
    arabizi_nums = sum(1 for c in text if c in '3579')
    total_alpha = arabic_chars + latin_chars

    if total_alpha == 0:
        return 3  # UNKNOWN

    arabic_ratio = arabic_chars / total_alpha
    latin_ratio = latin_chars / total_alpha

    # Check for Arabizi (number-letters present)
    if arabizi_nums >= 1 and latin_chars > arabic_chars:
        return 0  # GENZ_ARABIZI

    # Check for Arabic dominant
    if arabic_ratio > 0.75:
        return 2  # ARABIC_DOM

    # Check for mixed
    if 0.25 <= arabic_ratio <= 0.75:
        return 1  # MIXED

    # Latin dominant without Arabizi numbers
    if latin_ratio > 0.75 and arabizi_nums == 0:
        return 3  # UNKNOWN (probably not Derja)

    return 1  # Default to MIXED

# Test the register detection
test_cases = [
    "esma3ni ya bro, chnia el 7keya?",
    "كيفاش الحال خويا؟ لاباس عليك",
    "هاني mech نجي because عندي خدمة",
    "This is pure English text",
]

print("Register Detection Test:")
for text in test_cases:
    reg = detect_register(text)
    print(f"  [{REGISTERS[reg]:12}] {text[:50]}...")

Register Detection Test:
  [GENZ_ARABIZI] esma3ni ya bro, chnia el 7keya?...
  [ARABIC_DOM  ] كيفاش الحال خويا؟ لاباس عليك...
  [MIXED       ] هاني mech نجي because عندي خدمة...
  [UNKNOWN     ] This is pure English text...


## 6. Deduplication

In [7]:
class MinHashDeduplicator:
    """Semantic deduplication using MinHash LSH."""

    def __init__(self, threshold: float = 0.85, num_perm: int = 128):
        self.threshold = threshold
        self.num_perm = num_perm
        self.lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)
        self.seen_hashes = set()
        self.n_total = 0
        self.n_duplicates = 0

    def _get_minhash(self, text: str) -> MinHash:
        """Create MinHash signature for text."""
        m = MinHash(num_perm=self.num_perm)

        # Use character 3-grams for better matching across scripts
        text_clean = text.lower().replace(' ', '')
        for i in range(len(text_clean) - 2):
            shingle = text_clean[i:i+3]
            m.update(shingle.encode('utf-8'))

        return m

    def is_duplicate(self, text: str, doc_id: str) -> bool:
        """Check if text is a duplicate and add to index if not."""
        self.n_total += 1

        # Quick exact hash check first
        text_hash = hashlib.md5(text.encode('utf-8')).hexdigest()
        if text_hash in self.seen_hashes:
            self.n_duplicates += 1
            return True
        self.seen_hashes.add(text_hash)

        # Skip MinHash for very short texts
        if len(text) < 30:
            return False

        # MinHash similarity check
        m = self._get_minhash(text)

        # Query for similar documents
        result = self.lsh.query(m)
        if result:
            self.n_duplicates += 1
            return True

        # Add to index
        try:
            self.lsh.insert(doc_id, m)
        except ValueError:
            # Key already exists (shouldn't happen but handle gracefully)
            pass

        return False

    def report(self) -> None:
        """Print deduplication statistics."""
        kept = self.n_total - self.n_duplicates
        dup_rate = self.n_duplicates / self.n_total * 100 if self.n_total > 0 else 0
        print(f"\n📊 Deduplication: {self.n_duplicates:,} duplicates removed ({dup_rate:.1f}%)")
        print(f"   Kept: {kept:,} unique samples")

print("✅ MinHashDeduplicator class defined")

✅ MinHashDeduplicator class defined


## 7. Dataset Loaders

In [8]:
def extract_text_from_item(item: dict, text_keys: List[str]) -> Optional[str]:
    """Extract text from a dataset item, trying multiple keys."""
    for key in text_keys:
        if key in item:
            val = item[key]
            if isinstance(val, str) and val.strip():
                return val.strip()
            elif isinstance(val, list):
                # Join list elements
                texts = [str(t) for t in val if t]
                if texts:
                    return ' '.join(texts)
    return None


def load_hamzabouajila_corpus(max_samples: int = 300000) -> List[str]:
    """Load hamzabouajila/tunisian-derja-unified-raw-corpus (PRIMARY)."""
    print("\n📥 Loading hamzabouajila/tunisian-derja-unified-raw-corpus...")

    samples = []
    text_keys = ['text', 'content', 'sentence']

    try:
        ds = load_dataset("hamzabouajila/tunisian-derja-unified-raw-corpus",
                         split="train", streaming=True)

        for item in tqdm(ds, desc="hamzabouajila", total=max_samples):
            if len(samples) >= max_samples:
                break

            text = extract_text_from_item(item, text_keys)
            if text:
                samples.append(text)

    except Exception as e:
        print(f"   ⚠️ Error loading hamzabouajila: {e}")

    print(f"   ✅ Loaded {len(samples):,} samples")
    return samples


def load_linagora_dataset(max_per_config: int = 50000) -> List[str]:
    """Load linagora/Tunisian_Derja_Dataset (SECONDARY)."""
    print("\n📥 Loading linagora/Tunisian_Derja_Dataset...")

    samples = []
    configs = ["Derja_tunsi", "TunBERT", "TunSwitchTunisiaOnly"]
    text_keys = ['text', 'sentence', 'content', 'source', 'target', 'sent1', 'sent2']

    for cfg in configs:
        try:
            print(f"   Loading config: {cfg}...")
            ds = load_dataset("linagora/Tunisian_Derja_Dataset", cfg,
                             split="train", streaming=True)

            count = 0
            for item in ds:
                if count >= max_per_config:
                    break

                text = extract_text_from_item(item, text_keys)
                if text:
                    samples.append(text)
                    count += 1

            print(f"     Collected {count:,} from {cfg}")

        except Exception as e:
            print(f"     ⚠️ Failed to load {cfg}: {e}")

    print(f"   ✅ Total from linagora: {len(samples):,} samples")
    return samples


def load_tedxtn() -> List[str]:
    """Load fbougares/TEDxTN (HIGH QUALITY)."""
    print("\n📥 Loading fbougares/TEDxTN...")

    samples = []
    text_keys = ['text', 'transcription', 'transcript', 'aeb']

    try:
        ds = load_dataset("fbougares/TEDxTN", split="train")

        for item in ds:
            text = extract_text_from_item(item, text_keys)
            if text:
                samples.append(text)

    except Exception as e:
        print(f"   ⚠️ Error loading TEDxTN: {e}")

    print(f"   ✅ Loaded {len(samples):,} samples")
    return samples


def load_tunizi() -> List[str]:
    """Load chaymafourati/tunizi (ARABIZI)."""
    print("\n📥 Loading chaymafourati/tunizi...")

    samples = []
    text_keys = ['text', 'sentence', 'review']

    try:
        ds = load_dataset("chaymafourati/tunizi", split="train")

        for item in ds:
            text = extract_text_from_item(item, text_keys)
            if text:
                samples.append(text)

    except Exception as e:
        print(f"   ⚠️ Error loading tunizi: {e}")

    print(f"   ✅ Loaded {len(samples):,} samples")
    return samples


def load_arbml_tweets(max_samples: int = 30000) -> List[str]:
    """Load arbml/Tunisian_Dialect_Corpus (TWEETS)."""
    print("\n📥 Loading arbml/Tunisian_Dialect_Corpus...")

    samples = []
    text_keys = ['Tweet', 'text', 'content']

    try:
        ds = load_dataset("arbml/Tunisian_Dialect_Corpus", split="train")

        for item in ds:
            if len(samples) >= max_samples:
                break

            text = extract_text_from_item(item, text_keys)
            if text:
                samples.append(text)

    except Exception as e:
        print(f"   ⚠️ Error loading arbml: {e}")

    print(f"   ✅ Loaded {len(samples):,} samples")
    return samples


print("✅ Dataset loaders defined")

✅ Dataset loaders defined


In [9]:
def load_khaled123_harsh(max_samples: int = 80000) -> List[str]:
    """
    Load khaled123/Tunisian_Dialectic_English_Derja with HARSH filtering.

    This dataset has known issues:
    - Heavy English contamination
    - Unethical/offensive content
    - Low overall quality

    We apply extra strict filtering to extract only high-quality Derja.
    """
    print("\n📥 Loading khaled123/Tunisian_Dialectic_English_Derja (HARSH MODE)...")
    print("   ⚠️ Applying extra strict filtering for this dataset")

    samples = []
    text_keys = ['text', 'derja', 'tunisian', 'content', 'sentence']

    # Create harsh-mode cleaner for this dataset
    harsh_cleaner = DerjaTextCleaner(config, harsh_mode=True)

    try:
        # Try different possible configurations
        ds = None
        for split in ['train', 'test', 'validation']:
            try:
                ds = load_dataset("khaled123/Tunisian_Dialectic_English_Derja",
                                 split=split, streaming=True)
                print(f"   Using split: {split}")
                break
            except:
                continue

        if ds is None:
            # Try without specifying split
            ds = load_dataset("khaled123/Tunisian_Dialectic_English_Derja", streaming=True)
            if isinstance(ds, dict):
                ds = list(ds.values())[0]

        processed = 0
        for item in tqdm(ds, desc="khaled123 (harsh)", total=max_samples*3):
            if len(samples) >= max_samples:
                break

            processed += 1

            text = extract_text_from_item(item, text_keys)
            if not text:
                continue

            # Apply harsh cleaning
            cleaned = harsh_cleaner.clean(text)
            if cleaned:
                samples.append(cleaned)

        harsh_cleaner.report()

    except Exception as e:
        print(f"   ⚠️ Error loading khaled123: {e}")
        import traceback
        traceback.print_exc()

    print(f"   ✅ Extracted {len(samples):,} clean samples from khaled123")
    return samples

print("✅ Harsh khaled123 loader defined")

✅ Harsh khaled123 loader defined


## 8. Main Data Processing Pipeline

In [10]:
def run_full_pipeline():
    """Run the complete data preparation pipeline."""

    print("\n" + "=" * 70)
    print("🚀 NANOLM-DERJA V4: DATA PREPARATION PIPELINE")
    print("=" * 70)

    # Initialize cleaner and deduplicator
    cleaner = DerjaTextCleaner(config, harsh_mode=False)
    deduper = MinHashDeduplicator(
        threshold=config.MINHASH_THRESHOLD,
        num_perm=config.MINHASH_NUM_PERM
    )

    all_samples = []
    doc_id = 0

    # === PHASE 1: Load and process core datasets ===
    print("\n" + "-" * 50)
    print("PHASE 1: Loading Core Datasets")
    print("-" * 50)

    datasets_raw = {
        'hamzabouajila': load_hamzabouajila_corpus(300000),
        'linagora': load_linagora_dataset(50000),
        'tedxtn': load_tedxtn(),
        'tunizi': load_tunizi(),
        'arbml': load_arbml_tweets(30000),
    }

    # Process each dataset
    print("\n" + "-" * 50)
    print("PHASE 2: Cleaning Core Datasets")
    print("-" * 50)

    for ds_name, raw_samples in datasets_raw.items():
        print(f"\n🔄 Processing {ds_name}...")
        kept = 0

        for text in tqdm(raw_samples, desc=ds_name):
            cleaned = cleaner.clean(text)
            if cleaned and not deduper.is_duplicate(cleaned, f"{ds_name}_{doc_id}"):
                register = detect_register(cleaned)
                all_samples.append({
                    'text': cleaned,
                    'source': ds_name,
                    'register_id': register,
                })
                doc_id += 1
                kept += 1

        print(f"   Kept {kept:,} samples from {ds_name}")

    # === PHASE 3: Load and process khaled123 with HARSH filtering ===
    print("\n" + "-" * 50)
    print("PHASE 3: Processing khaled123 (HARSH MODE)")
    print("-" * 50)

    khaled_samples = load_khaled123_harsh(config.KHALED123_MAX_SAMPLES)

    # Deduplicate against existing samples
    print("\n🔄 Deduplicating khaled123 against core datasets...")
    khaled_kept = 0

    for text in tqdm(khaled_samples, desc="khaled123 dedup"):
        if not deduper.is_duplicate(text, f"khaled123_{doc_id}"):
            register = detect_register(text)
            all_samples.append({
                'text': text,
                'source': 'khaled123',
                'register_id': register,
            })
            doc_id += 1
            khaled_kept += 1

    print(f"   Kept {khaled_kept:,} unique samples from khaled123")

    # === PHASE 4: Statistics and Reporting ===
    print("\n" + "-" * 50)
    print("PHASE 4: Final Statistics")
    print("-" * 50)

    cleaner.report()
    deduper.report()

    # Source distribution
    source_counts = Counter(s['source'] for s in all_samples)
    print("\n📊 Source Distribution:")
    for source, count in source_counts.most_common():
        pct = count / len(all_samples) * 100
        print(f"   {source}: {count:,} ({pct:.1f}%)")

    # Register distribution
    register_counts = Counter(s['register_id'] for s in all_samples)
    print("\n📊 Register Distribution:")
    for reg_id, count in sorted(register_counts.items()):
        pct = count / len(all_samples) * 100
        print(f"   {REGISTERS[reg_id]}: {count:,} ({pct:.1f}%)")

    return all_samples

print("✅ Pipeline function defined")

✅ Pipeline function defined


In [11]:
# Run the full pipeline
all_samples = run_full_pipeline()


🚀 NANOLM-DERJA V4: DATA PREPARATION PIPELINE

--------------------------------------------------
PHASE 1: Loading Core Datasets
--------------------------------------------------

📥 Loading hamzabouajila/tunisian-derja-unified-raw-corpus...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

hamzabouajila:   0%|          | 0/300000 [00:00<?, ?it/s]

   ✅ Loaded 300,000 samples

📥 Loading linagora/Tunisian_Derja_Dataset...
   Loading config: Derja_tunsi...


...

   Using split: train


khaled123 (harsh):   0%|          | 0/240000 [00:00<?, ?it/s]


CLEANING STATISTICS
  ✅ kept: 80,000 (41.5%)
  ❌ low_arabic: 68,917 (35.8%)
  ❌ no_derja_markers: 34,127 (17.7%)
  ❌ repetitive: 5,416 (2.8%)
  ❌ nsfw: 2,385 (1.2%)
  ❌ too_short: 1,453 (0.8%)
  ❌ msa: 254 (0.1%)
  ❌ emoji_spam: 26 (0.0%)
--------------------------------------------------
  Total processed: 192,578
  Keep rate: 41.5%
   ✅ Extracted 80,000 clean samples from khaled123

🔄 Deduplicating khaled123 against core datasets...


khaled123 dedup:   0%|          | 0/80000 [00:00<?, ?it/s]

   Kept 75,919 unique samples from khaled123

--------------------------------------------------
PHASE 4: Final Statistics
--------------------------------------------------

CLEANING STATISTICS
  ✅ kept: 311,346 (70.3%)
  ❌ too_short: 85,521 (19.3%)
  ❌ low_arabic: 30,073 (6.8%)
  ❌ msa: 7,558 (1.7%)
  ❌ nsfw: 4,490 (1.0%)
  ❌ repetitive: 3,962 (0.9%)
  ❌ emoji_spam: 87 (0.0%)
--------------------------------------------------
  Total processed: 443,037
  Keep rate: 70.3%

📊 Deduplication: 43,856 duplicates removed (11.2%)
   Kept: 347,490 unique samples

📊 Source Distribution:
   hamzabouajila: 215,080 (61.9%)
   khaled123: 75,919 (21.8%)
   linagora: 47,349 (13.6%)
   arbml: 9,142 (2.6%)

📊 Register Distribution:
   GENZ_ARABIZI: 58,519 (16.8%)
   MIXED: 4,250 (1.2%)
   ARABIC_DOM: 284,608 (81.9%)
   UNKNOWN: 113 (0.0%)


## 9. Create Train/Validation Splits

In [12]:
def create_splits(samples: List[dict], val_ratio: float = 0.05) -> Tuple[List[dict], List[dict]]:
    """Create stratified train/validation splits by register."""

    print("\n" + "=" * 50)
    print("Creating Train/Validation Splits")
    print("=" * 50)

    # Shuffle
    random.shuffle(samples)

    # Split by register for stratification
    by_register = {}
    for s in samples:
        reg = s['register_id']
        if reg not in by_register:
            by_register[reg] = []
        by_register[reg].append(s)

    train_samples = []
    val_samples = []

    for reg_id, reg_samples in by_register.items():
        n_val = max(1, int(len(reg_samples) * val_ratio))

        val_samples.extend(reg_samples[:n_val])
        train_samples.extend(reg_samples[n_val:])

        print(f"   {REGISTERS[reg_id]}: {len(reg_samples)-n_val:,} train, {n_val:,} val")

    # Final shuffle
    random.shuffle(train_samples)
    random.shuffle(val_samples)

    print(f"\n📊 Final Split:")
    print(f"   Train: {len(train_samples):,} samples")
    print(f"   Val: {len(val_samples):,} samples")

    return train_samples, val_samples

train_samples, val_samples = create_splits(all_samples, config.VAL_RATIO)


Creating Train/Validation Splits
   ARABIC_DOM: 270,378 train, 14,230 val
   GENZ_ARABIZI: 55,594 train, 2,925 val
   MIXED: 4,038 train, 212 val
   UNKNOWN: 108 train, 5 val

📊 Final Split:
   Train: 330,118 samples
   Val: 17,372 samples


## 10. Save Datasets

In [13]:
def save_to_jsonl(samples: List[dict], filepath: str) -> None:
    """Save samples to JSONL format."""
    with open(filepath, 'w', encoding='utf-8') as f:
        for sample in samples:
            f.write(json.dumps(sample, ensure_ascii=False) + '\n')

    file_size = os.path.getsize(filepath) / (1024 * 1024)  # MB
    print(f"   Saved {filepath} ({file_size:.1f} MB)")

# Save datasets
print("\n💾 Saving datasets...")

train_path = os.path.join(config.OUTPUT_DIR, config.TRAIN_FILE)
val_path = os.path.join(config.OUTPUT_DIR, config.VAL_FILE)

save_to_jsonl(train_samples, train_path)
save_to_jsonl(val_samples, val_path)

print("\n✅ Datasets saved successfully!")


💾 Saving datasets...
   Saved derja_data_v4/train.jsonl (272.5 MB)
   Saved derja_data_v4/val.jsonl (14.2 MB)

✅ Datasets saved successfully!


## 11. Manual Validation Audit

**CRITICAL STEP**: Review a random sample of tagged data before training.

Run this cell and manually check:
1. Is the register tag correct?
2. Is the text actually Tunisian Derja?
3. Is there any offensive content that slipped through?

In [14]:
def manual_validation_audit(samples: List[dict], n_samples: int = 50) -> None:
    """
    Display random samples for manual validation.

    Review each sample and check:
    1. Register tag accuracy
    2. Text quality (is it Derja?)
    3. No offensive content
    """

    print("\n" + "=" * 70)
    print("MANUAL VALIDATION AUDIT")
    print("=" * 70)
    print("\nReview each sample and note any issues.")
    print("Press Enter to continue to next sample, or 'q' to quit.\n")

    indices = random.sample(range(len(samples)), min(n_samples, len(samples)))

    issues_found = 0

    for i, idx in enumerate(indices):
        sample = samples[idx]
        text = sample['text']
        source = sample['source']
        register = REGISTERS[sample['register_id']]

        print("-" * 70)
        print(f"Sample {i+1}/{n_samples} (from {source})")
        print(f"Register: [{register}]")
        print("-" * 70)
        print(f"\n{text[:500]}{'...' if len(text) > 500 else ''}\n")

        # In notebook, user reviews manually
        # For automated check, we just display

    print("\n" + "=" * 70)
    print("AUDIT COMPLETE")
    print("=" * 70)
    print("\nIf you found issues, consider:")
    print("1. Adjusting the cleaning thresholds")
    print("2. Adding terms to the blocklist")
    print("3. Re-running the pipeline")

# Run audit on training samples
manual_validation_audit(train_samples, n_samples=30)


MANUAL VALIDATION AUDIT

Review each sample and note any issues.
Press Enter to continue to next sample, or 'q' to quit.

----------------------------------------------------------------------
Sample 1/30 (from linagora)
Register: [ARABIC_DOM]
----------------------------------------------------------------------

ربي آروي قلوبنآ بك ل آمنية نحلم بهآ

----------------------------------------------------------------------
Sample 2/30 (from hamzabouajila)
Register: [ARABIC_DOM]
----------------------------------------------------------------------

أعطيها سوساتها موش فوقك

----------------------------------------------------------------------
Sample 3/30 (from khaled123)
Register: [GENZ_ARABIZI]
----------------------------------------------------------------------

Sure! Let's break down the translation step-by-step. ### Original Text: يوقفوك كيما قلت أنت يسكروا فوروم فيه عضو و يسكروا و والا حاجة هكا والا هكا الفاهم يفهم مش يسكرو الموضوع ### Step-by-Step Translation: 1. **يوقفوك كيما قل

In [15]:
# Specifically audit khaled123 samples
print("\n🔍 Auditing khaled123 samples specifically...\n")

khaled_samples = [s for s in train_samples if s['source'] == 'khaled123']
print(f"Total khaled123 samples in training set: {len(khaled_samples):,}")

if khaled_samples:
    manual_validation_audit(khaled_samples, n_samples=20)


🔍 Auditing khaled123 samples specifically...

Total khaled123 samples in training set: 72,198

MANUAL VALIDATION AUDIT

Review each sample and note any issues.
Press Enter to continue to next sample, or 'q' to quit.

----------------------------------------------------------------------
Sample 1/20 (from khaled123)
Register: [GENZ_ARABIZI]
----------------------------------------------------------------------

Sure! Let's break down the translation step-by-step and then I'll explain the choices made. ### Step-by-Step Translation: 1. **احتجوا على مشروع قانون** - **Translation:** "They protested against the draft law" - **Explanation:** "احتجوا" means "they protested," and "مشروع قانون" translates to "draft law." This phrase sets the context of a protest regarding legislation. 2. **تفويض رئيس الجمهورية المؤقت** - **Translation:** "to authorize the interim President" - **Explanation:** "تفويض" means "to ...

----------------------------------------------------------------------
Sample 2/

## 12. Length Statistics

In [16]:
# Compute and display length statistics
def compute_length_stats(samples: List[dict], name: str) -> None:
    """Compute and display text length statistics."""
    lengths = [len(s['text']) for s in samples]
    word_counts = [len(s['text'].split()) for s in samples]

    print(f"\n📊 {name} Statistics:")
    print(f"   Samples: {len(samples):,}")
    print(f"   Character length: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)/len(lengths):.0f}")
    print(f"   Word count: min={min(word_counts)}, max={max(word_counts)}, avg={sum(word_counts)/len(word_counts):.1f}")

    # Estimate tokens (rough: ~4 chars per token for Arabic)
    total_chars = sum(lengths)
    est_tokens = total_chars / 4
    print(f"   Total characters: {total_chars:,}")
    print(f"   Estimated tokens: ~{est_tokens/1e6:.1f}M")

compute_length_stats(train_samples, "Training Set")
compute_length_stats(val_samples, "Validation Set")


📊 Training Set Statistics:
   Samples: 330,118
   Character length: min=20, max=2000, avg=561
   Word count: min=1, max=465, avg=93.3
   Total characters: 185,282,711
   Estimated tokens: ~46.3M

📊 Validation Set Statistics:
   Samples: 17,372
   Character length: min=20, max=2000, avg=555
   Word count: min=1, max=403, avg=92.3
   Total characters: 9,637,756
   Estimated tokens: ~2.4M


## 13. Push to HuggingFace (Optional)

In [17]:
# Optional: Push dataset to HuggingFace Hub
# Uncomment and fill in your details to use

PUSH_TO_HF = False  # Set to True to push
HF_USERNAME = "your-username"
HF_DATASET_NAME = "nanolm-derja-v4-data"

if PUSH_TO_HF:
    from huggingface_hub import login, HfApi

    # Login (will prompt for token)
    login()

    # Create dataset
    train_ds = Dataset.from_list(train_samples)
    val_ds = Dataset.from_list(val_samples)

    ds_dict = DatasetDict({
        'train': train_ds,
        'validation': val_ds,
    })

    # Push
    repo_id = f"{HF_USERNAME}/{HF_DATASET_NAME}"
    ds_dict.push_to_hub(repo_id)
    print(f"\n✅ Dataset pushed to: https://huggingface.co/datasets/{repo_id}")
else:
    print("\n⏭️ Skipping HuggingFace push (PUSH_TO_HF = False)")


⏭️ Skipping HuggingFace push (PUSH_TO_HF = False)


## 14. Summary & Next Steps

In [18]:
print("\n" + "=" * 70)
print("✅ DATA PREPARATION COMPLETE")
print("=" * 70)

print(f"\n📁 Output files:")
print(f"   {train_path}")
print(f"   {val_path}")

print(f"\n📊 Dataset Summary:")
print(f"   Total training samples: {len(train_samples):,}")
print(f"   Total validation samples: {len(val_samples):,}")

print("\n🔜 Next Steps:")
print("   1. Download the output files to your local machine")
print("   2. Run Notebook 2 (Tokenizer Training) on Colab")
print("   3. Upload data + tokenizer to Kaggle for training")

print("\n💡 Tips:")
print("   - If you found issues in the audit, adjust thresholds and re-run")
print("   - The khaled123 samples should be <20% of total data")
print("   - Target ~400k-600k total samples for good pre-training")


✅ DATA PREPARATION COMPLETE

📁 Output files:
   derja_data_v4/train.jsonl
   derja_data_v4/val.jsonl

📊 Dataset Summary:
   Total training samples: 330,118
   Total validation samples: 17,372

🔜 Next Steps:
   1. Download the output files to your local machine
   2. Run Notebook 2 (Tokenizer Training) on Colab
   3. Upload data + tokenizer to Kaggle for training

💡 Tips:
   - If you found issues in the audit, adjust thresholds and re-run
   - The khaled123 samples should be <20% of total data
   - Target ~400k-600k total samples for good pre-training


In [19]:
# Download files (for Colab)
try:
    from google.colab import files

    print("📥 Downloading output files...")
    files.download(train_path)
    files.download(val_path)
except:
    print("💡 Not running on Colab. Files are saved locally in:", config.OUTPUT_DIR)

📥 Downloading output files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>